# Нейросетевые модели

## Часть 3.1: Архитектуры для работы с последовательностями данных - LSTM

### 1. Данные для обучения

__CMAPSS (Commercial Modular Aero-Propulsion System Simulation)__ — это широко известный набор данных для задач прогнозирования остаточного ресурса (RUL, Remaining Useful Life) в области предиктивной аналитики и технического обслуживания. Обычно его называют CMAPSS Data или N-CMAPSS (для более новых версий). Чаще всего он ассоциируется с соревнованием PHM08 (Prognostics and Health Management, 2008).

Основные характеристики:
- __Источник:__ NASA
- __Объект:__ Моделирование работы турбовентиляторного двигателя
- __Задача:__ Несколько обучающих временных рядов (двигатели), которые работают от начала жизни до отказа, и тестовых рядов, которые обрываются до отказа. Нужно предсказать, сколько еще циклов проработает тестовый двигатель
- __Данные:__ 26 столбцов (включая номер двигателя, время, 3 эксплуатационных режима и 21 датчик — температуры, давления, обороты и т.д.)

In [ ]:
!ls -la /home/jovyan/__DATA/GZP/cmapssdata

Данные моделируют поведение авиадвигателей в условиях деградации. Каждый сценарий имитирует работу двигателя с постепенным ухудшением его характеристик:
- __FD001__ (один режим работы, один отказ)
- __FD002__ (множество режимов работы, один отказ)
- __FD003__ (один режим работы, множество отказов)
- __FD004__ (множество режимов работы, множество отказов)

Структура файлов:
- train_FDxxx.txt — данные для обучения.
- test_FDxxx.txt — данные для тестирования.
- RUL_FDxxx.txt — оставшийся ресурс двигателя для тестовых данных.

Составляющие данных:
1. ID двигателя: уникальный идентификатор каждого двигателя.
2. Цикл: количество циклов, пройденных двигателем (симуляция времени).
3. Операционные настройки: три переменные, описывающие условия эксплуатации
двигателя.
4. Сенсорные данные: данные с 21 датчика, регистрирующих состояние двигателя
(например, температура, давление, скорость вращения).

### 2. Решение задачи с использованием рекуррентных нейросетей

### Суть применения LSTM для прогнозирования RUL

**Проблема:** Нужно предсказать, сколько циклов осталось до отказа двигателя, имея временные ряды сенсоров.

**Почему LSTM, а не обычные методы?**

1. **Временная зависимость**
- Отказ не случается мгновенно — это **процесс деградации**
- LSTM помнит, что было 10, 20, 50 циклов назад
- Обычные методы (Random Forest, XGBoost) видят только текущий момент
2. **Долгосрочная память**
```
Цикл 1:  Сенсоры в норме
Цикл 50: Начало деградации  ← LSTM помнит
Цикл 100: Явный отказ
```
- LSTM решает проблему "забывания" через специальные вентили (gate mechanism)
3. **Последовательность → Одно значение**
```
Вход: [30 циклов × 21 сенсор] → LSTM → Выход: RUL (одно число)
```
4. **Ключевое преимущество**
- **RNN обычный:** забывает ранние паттерны
- **LSTM:** сохраняет важную информацию через "ячейку памяти"

**Простая аналогия:** Обычная нейросеть — смотрит на **фото** текущего состояния двигателя. LSTM — смотрит **видео** всей истории эксплуатации

**Итог:** LSTM идеален для CMAPSS, потому что отказ двигателя — это временной процесс, а не мгновенное событие.

#### 2.1. Библиотеки для решения задачи

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Bidirectional, Attention, Concatenate, Lambda
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.figsize'] = (12, 6)

#### 2.2. Загрузка данных

In [ ]:
DATA_PATH = '/home/jovyan/__DATA/GZP/cmapssdata'

# Имена колонок (для всех датасетов)
col_names = ['unit_number', 'time_cycles'] + \
            [f'op_setting_{i}' for i in range(1, 4)] + \
            [f'sensor_{i}' for i in range(1, 22)]

def load_cmapss_dataset(dataset_id):
    """
    Загрузка одного датасета CMAPSS
    dataset_id: 'FD001', 'FD002', 'FD003', 'FD004'
    """
    train_file = f'{DATA_PATH}/train_{dataset_id}.txt'
    test_file = f'{DATA_PATH}/test_{dataset_id}.txt'
    rul_file = f'{DATA_PATH}/RUL_{dataset_id}.txt'
    
    # Загрузка данных
    train_df = pd.read_csv(train_file, sep='\s+', header=None, names=col_names)
    test_df = pd.read_csv(test_file, sep='\s+', header=None, names=col_names)
    rul_df = pd.read_csv(rul_file, sep='\s+', header=None, names=['rul'])
    
    # Добавляем мета-информацию
    train_df['dataset'] = dataset_id
    test_df['dataset'] = dataset_id
    
    return train_df, test_df, rul_df

# Загрузка всех датасетов
datasets = {}
print("Загрузка данных...")
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    train, test, rul = load_cmapss_dataset(ds_id)
    datasets[ds_id] = {'train': train, 'test': test, 'rul': rul}
    print(f"{ds_id}: train={train.shape}, test={test.shape}, rul={len(rul)}")

#### 2.3. Описание данных

In [ ]:
dataset_info = pd.DataFrame({
    'Dataset': ['FD001', 'FD002', 'FD003', 'FD004'],
    'Train engines': [datasets[d]['train']['unit_number'].nunique() for d in ['FD001','FD002','FD003','FD004']],
    'Test engines': [datasets[d]['test']['unit_number'].nunique() for d in ['FD001','FD002','FD003','FD004']],
    'Operating conditions': [1, 6, 1, 6],
    'Fault modes': [1, 1, 2, 2],
    'Description': [
        'Single condition, single fault',
        'Multiple conditions, single fault',
        'Single condition, multiple faults',
        'Multiple conditions, multiple faults'
    ]
})
print(dataset_info.to_string(index=False))

# Визуализация распределения жизненных циклов
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, ds_id in enumerate(['FD001', 'FD002', 'FD003', 'FD004']):
    train_df = datasets[ds_id]['train']
    max_cycles = train_df.groupby('unit_number')['time_cycles'].max()
    
    axes[idx].hist(max_cycles, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].axvline(max_cycles.mean(), color='r', linestyle='--', linewidth=2, 
                      label=f'Среднее: {max_cycles.mean():.0f}')
    axes[idx].set_xlabel('Жизненный цикл (циклы)')
    axes[idx].set_ylabel('Количество двигателей')
    axes[idx].set_title(f'{ds_id}: Распределение жизненных циклов')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dataset_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

#### 2.4. Детальный анализ сенсоров

In [ ]:
def analyze_sensors(train_df, dataset_id):
    """Анализ сенсоров для датасета"""
    sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
    
    # Статистика сенсоров
    sensor_stats = train_df[sensor_cols].describe().T
    sensor_stats['variance'] = train_df[sensor_cols].var()
    sensor_stats['trend_corr'] = [train_df[sensor].corr(train_df['time_cycles']) for sensor in sensor_cols]
    
    # Определяем информативные сенсоры (высокая корреляция со временем)
    informative = sensor_stats[abs(sensor_stats['trend_corr']) > 0.3].index.tolist()
    
    return sensor_stats, informative

# Анализ для каждого датасета
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, ds_id in enumerate(['FD001', 'FD002', 'FD003', 'FD004']):
    train_df = datasets[ds_id]['train']
    sensor_stats, informative = analyze_sensors(train_df, ds_id)
    
    # Визуализация корреляций
    correlations = sensor_stats['trend_corr']
    colors = ['green' if abs(c) > 0.3 else 'red' for c in correlations]
    axes[idx].bar(range(1, 22), correlations.values, color=colors, alpha=0.7)
    axes[idx].axhline(y=0.3, color='g', linestyle='--', alpha=0.5, label='Порог (0.3)')
    axes[idx].axhline(y=-0.3, color='g', linestyle='--', alpha=0.5)
    axes[idx].set_xlabel('Номер сенсора')
    axes[idx].set_ylabel('Корреляция с временем')
    axes[idx].set_title(f'{ds_id}: Корреляция сенсоров с деградацией')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_xticks(range(1, 22, 2))
    
    print(f"\n{ds_id}: Информативные сенсоры ({len(informative)}): {informative[:5]}...")

plt.tight_layout()
plt.savefig('sensors_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

#### 2.5. Предобработка данных

In [ ]:
def add_rul_labels(df, max_rul=125, use_max_rul=True):
    """
    Добавление целевой переменной RUL для обучающих данных
    """
    # Группируем по двигателям
    df['rul'] = df.groupby('unit_number')['time_cycles'].transform(
        lambda x: x.max() - x
    )
    # Ограничиваем максимальное значение RUL (опционально)
    if use_max_rul:
        df['rul'] = df['rul'].clip(upper=max_rul)
    return df

def select_features(train_df, test_df, correlation_threshold=0.3):
    """
    Выбор информативных сенсоров на основе корреляции
    """
    sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
    op_setting_cols = [f'op_setting_{i}' for i in range(1, 4)]
    
    # Расчет корреляции с временем
    correlations = {}
    for sensor in sensor_cols:
        corr = abs(train_df[sensor].corr(train_df['time_cycles']))
        correlations[sensor] = corr
    
    # Выбор сенсоров с высокой корреляцией
    selected_sensors = [s for s in sensor_cols if correlations[s] > correlation_threshold]
    
    # Всегда включаем operational settings
    feature_cols = op_setting_cols + selected_sensors
    
    print(f"Выбрано {len(feature_cols)} признаков: {feature_cols[:5]}...")
    
    return feature_cols

def prepare_sequences_train(data, seq_length=30, feature_cols=None):
    """
    Подготовка последовательностей для LSTM (для обучающих данных с RUL)
    """
    if feature_cols is None:
        feature_cols = [col for col in data.columns if 'sensor' in col or 'op_setting' in col]
    
    X, y, unit_ids = [], [], []
    
    for unit_id in data['unit_number'].unique():
        unit_data = data[data['unit_number'] == unit_id].sort_values('time_cycles')
        
        if len(unit_data) < seq_length:
            continue
            
        features = unit_data[feature_cols].values
        
        for i in range(len(unit_data) - seq_length + 1):
            X.append(features[i:i+seq_length])
            y.append(unit_data['rul'].iloc[i+seq_length-1])
            unit_ids.append(unit_id)
    
    return np.array(X), np.array(y), np.array(unit_ids)

def prepare_sequences_test(data, seq_length=30, feature_cols=None):
    """
    Подготовка последовательностей для LSTM (для тестовых данных без RUL)
    """
    if feature_cols is None:
        feature_cols = [col for col in data.columns if 'sensor' in col or 'op_setting' in col]
    
    X, unit_ids = [], []
    
    for unit_id in data['unit_number'].unique():
        unit_data = data[data['unit_number'] == unit_id].sort_values('time_cycles')
        
        if len(unit_data) < seq_length:
            continue
            
        features = unit_data[feature_cols].values
        
        # Для тестовых данных берем все возможные последовательности
        for i in range(len(unit_data) - seq_length + 1):
            X.append(features[i:i+seq_length])
            unit_ids.append(unit_id)
    
    return np.array(X), np.array(unit_ids)

#### 2.6. Модель

In [ ]:
def build_improved_lstm_model(input_shape):
    """
    Улучшенная LSTM модель с Bidirectional и Attention (исправленная версия)
    """
    inputs = Input(shape=input_shape)
    
    # Первый Bidirectional LSTM слой
    x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
    x = Dropout(0.2)(x)
    
    # Второй LSTM слой
    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(0.2)(x)
    
    # Attention механизм (исправленная версия)
    # Используем Dense слой для вычисления весов внимания
    attention_weights = Dense(1, activation='tanh')(x)
    attention_weights = Dense(1, activation='sigmoid')(attention_weights)
    
    # Нормализация весов с помощью Lambda слоя
    def normalize_attention(x):
        return x / (K.sum(x, axis=1, keepdims=True) + K.epsilon())
    
    attention_weights = Lambda(normalize_attention)(attention_weights)
    
    # Применяем веса внимания
    x = tf.multiply(x, attention_weights)
    
    # Global average pooling по временному измерению
    x = tf.reduce_mean(x, axis=1)
    
    # Полносвязные слои
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.1)(x)
    
    output = Dense(1)(x)
    
    model = Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='huber',  # Более робастная loss-функция
        metrics=['mae']
    )
    
    return model

# Альтернативная более простая модель (на случай проблем с attention)
def build_simple_lstm_model(input_shape):
    """
    Простая LSTM модель (запасной вариант)
    """
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

def train_for_dataset(dataset_id, seq_len=30, max_rul=125, use_max_rul=True, use_simple_model=False):
    """
    Обучение модели для конкретного датасета
    """
    print(f"\n{'='*70}")
    print(f"ОБУЧЕНИЕ ДЛЯ {dataset_id}")
    print(f"{'='*70}")
    
    train_df = datasets[dataset_id]['train'].copy()
    test_df = datasets[dataset_id]['test'].copy()
    rul_true = datasets[dataset_id]['rul']['rul'].values
    
    print(f"Количество тестовых двигателей: {len(rul_true)}")
    
    # Добавление RUL меток только для обучающих данных
    train_df = add_rul_labels(train_df, max_rul, use_max_rul)
    
    # Выбор признаков
    feature_cols = select_features(train_df, test_df, correlation_threshold=0.3)
    
    # Нормализация
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train_df[feature_cols])
    test_scaled = scaler.transform(test_df[feature_cols])
    
    train_df_scaled = train_df.copy()
    test_df_scaled = test_df.copy()
    train_df_scaled[feature_cols] = train_scaled
    test_df_scaled[feature_cols] = test_scaled
    
    # Подготовка последовательностей
    X_train, y_train, _ = prepare_sequences_train(train_df_scaled, seq_len, feature_cols)
    X_test, test_units = prepare_sequences_test(test_df_scaled, seq_len, feature_cols)
    
    print(f"Форма данных: X_train={X_train.shape}, y_train={y_train.shape}, X_test={X_test.shape}")
    
    if len(X_train) == 0:
        print(f"Ошибка: Недостаточно данных для обучения в {dataset_id}")
        return None, None, None, None, None, None
    
    # Разделение на train/validation
    split_idx = int(0.8 * len(X_train))
    X_train_seq, X_val_seq = X_train[:split_idx], X_train[split_idx:]
    y_train_seq, y_val_seq = y_train[:split_idx], y_train[split_idx:]
    
    # Создание и обучение модели
    if use_simple_model:
        model = build_simple_lstm_model((seq_len, X_train.shape[2]))
    else:
        try:
            model = build_improved_lstm_model((seq_len, X_train.shape[2]))
        except Exception as e:
            print(f"Ошибка при создании улучшенной модели: {e}")
            print("Использую простую модель...")
            model = build_simple_lstm_model((seq_len, X_train.shape[2]))
    
    model.summary()
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    ]
    
    history = model.fit(
        X_train_seq, y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=100,
        batch_size=64,
        callbacks=callbacks,
        verbose=1
    )
    
    # Предсказания для тестовых двигателей
    y_pred_test = model.predict(X_test, verbose=0).flatten()
    y_pred_test = np.maximum(y_pred_test, 0)  # RUL не может быть отрицательным
    
    # Расчет RUL для каждого тестового двигателя (берем последнее предсказание)
    test_rul_pred = []
    for unit in test_df['unit_number'].unique():
        unit_preds = y_pred_test[test_units == unit]
        if len(unit_preds) > 0:
            test_rul_pred.append(unit_preds[-1])  # Берем последнее предсказание
        else:
            test_rul_pred.append(0)
    
    test_rul_pred = np.array(test_rul_pred)
    
    # Убеждаемся, что размерности совпадают
    if len(test_rul_pred) != len(rul_true):
        print(f"Предупреждение: размерности не совпадают! pred={len(test_rul_pred)}, true={len(rul_true)}")
        min_len = min(len(test_rul_pred), len(rul_true))
        test_rul_pred = test_rul_pred[:min_len]
        rul_true = rul_true[:min_len]
    
    # Метрики
    mse = mean_squared_error(rul_true, test_rul_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(rul_true, test_rul_pred)
    r2 = r2_score(rul_true, test_rul_pred)
    
    print(f"\nРезультаты для {dataset_id}:")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R²: {r2:.4f}")
    
    return model, history, rul_true, test_rul_pred, feature_cols, scaler

#### 2.7. Обучение модели

In [ ]:
results = {}
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, ds_id in enumerate(['FD001', 'FD002', 'FD003', 'FD004']):
    result = train_for_dataset(ds_id, seq_len=30, use_simple_model=True)  # Используем простую модель для надежности
    
    if result[0] is not None:
        model, history, true_rul, pred_rul, features, scaler = result
        results[ds_id] = {
            'model': model,
            'history': history,
            'true': true_rul,
            'pred': pred_rul,
            'features': features,
            'scaler': scaler,
            'rmse': np.sqrt(mean_squared_error(true_rul, pred_rul)),
            'mae': mean_absolute_error(true_rul, pred_rul)
        }
        
        # Визуализация результатов
        ax = axes[idx]
        ax.scatter(true_rul, pred_rul, alpha=0.6, s=30)
        
        max_val = max(true_rul.max(), pred_rul.max())
        ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Идеал')
        
        ax.set_xlabel('Истинный RUL')
        ax.set_ylabel('Предсказанный RUL')
        ax.set_title(f'{ds_id}: RMSE={results[ds_id]["rmse"]:.1f}, MAE={results[ds_id]["mae"]:.1f}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        print(f"Пропуск {ds_id} из-за ошибки")

plt.tight_layout()
plt.savefig('all_datasets_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Результаты

#### 3.1. Сравнительный анализ результатов

In [ ]:
if results:
    # Сводная таблица результатов
    comparison_df = pd.DataFrame({
        'Dataset': list(results.keys()),
        'RMSE': [results[ds]['rmse'] for ds in results],
        'MAE': [results[ds]['mae'] for ds in results],
        'Num_Features': [len(results[ds]['features']) for ds in results]
    })
    
    print("\n" + "="*70)
    print("СРАВНИТЕЛЬНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ")
    print("="*70)
    print(comparison_df.to_string(index=False))
    
    # Визуализация сравнения
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Метрики
    x = np.arange(len(comparison_df))
    width = 0.35
    
    axes[0].bar(x - width/2, comparison_df['RMSE'], width, label='RMSE', color='steelblue')
    axes[0].bar(x + width/2, comparison_df['MAE'], width, label='MAE', color='salmon')
    axes[0].set_xlabel('Dataset')
    axes[0].set_ylabel('Ошибка (циклы)')
    axes[0].set_title('Сравнение ошибок предсказания')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(comparison_df['Dataset'])
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Количество признаков
    axes[1].bar(comparison_df['Dataset'], comparison_df['Num_Features'], color='lightgreen', edgecolor='black')
    axes[1].set_xlabel('Dataset')
    axes[1].set_ylabel('Количество признаков')
    axes[1].set_title('Количество информативных сенсоров')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('comparison_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # ========================
    # ВИЗУАЛИЗАЦИЯ ПРОЦЕССА ОБУЧЕНИЯ
    # ========================
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, ds_id in enumerate(['FD001', 'FD002', 'FD003', 'FD004']):
        if ds_id in results:
            history = results[ds_id]['history']
            
            ax = axes[idx]
            ax.plot(history.history['loss'], label='Train Loss', linewidth=2)
            ax.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.set_title(f'{ds_id}: Процесс обучения')
            ax.legend()
            ax.grid(True, alpha=0.3)
            ax.set_yscale('log')
    
    plt.tight_layout()
    plt.savefig('training_processes.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # ========================
    # ПРИМЕР ПРОГНОЗА ДЛЯ КОНКРЕТНОГО ДВИГАТЕЛЯ
    # ========================
    
    def predict_specific_engine(dataset_id, engine_number):
        """
        Прогнозирование RUL для конкретного двигателя
        """
        if dataset_id not in results:
            print(f"Датасет {dataset_id} не обучен")
            return None
        
        test_df = datasets[dataset_id]['test']
        engine_data = test_df[test_df['unit_number'] == engine_number].sort_values('time_cycles')
        
        if len(engine_data) < 30:
            print(f"Недостаточно данных для двигателя {engine_number}")
            return None
        
        # Используем обученный scaler и features
        features = results[dataset_id]['features']
        scaler = results[dataset_id]['scaler']
        model = results[dataset_id]['model']
        
        # Нормализация и создание последовательности
        scaled_data = scaler.transform(engine_data[features])
        X_pred = scaled_data[-30:].reshape(1, 30, -1)
        
        # Предсказание
        pred_rul = model.predict(X_pred, verbose=0)[0][0]
        pred_rul = max(0, pred_rul)
        
        # Истинное значение (если известно)
        true_rul = None
        if engine_number <= len(datasets[dataset_id]['rul']):
            true_rul = datasets[dataset_id]['rul']['rul'].iloc[engine_number - 1]
        
        return pred_rul, true_rul
    
    # Примеры предсказаний
    print("\n" + "="*70)
    print("ПРИМЕРЫ ПРЕДСКАЗАНИЙ ДЛЯ КОНКРЕТНЫХ ДВИГАТЕЛЕЙ")
    print("="*70)
    
    for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
        if ds_id in results:
            pred, true = predict_specific_engine(ds_id, 1)
            if pred is not None:
                print(f"{ds_id} - Двигатель #1: Предсказанный RUL={pred:.0f}, Истинный RUL={true if true else 'N/A'}")
    
    # ========================
    # СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
    # ========================
    
    # Сохранение моделей и результатов
    import pickle
    
    for ds_id in results:
        # Сохранение модели
        results[ds_id]['model'].save(f'cmapss_model_{ds_id}.h5')
        
        # Сохранение метаданных
        metadata = {
            'features': results[ds_id]['features'],
            'scaler': results[ds_id]['scaler'],
            'rmse': results[ds_id]['rmse'],
            'mae': results[ds_id]['mae']
        }
        with open(f'cmapss_metadata_{ds_id}.pkl', 'wb') as f:
            pickle.dump(metadata, f)
    
    print("\n" + "="*70)
    print("МОДЕЛИ СОХРАНЕНЫ")
    print("="*70)
    print("Файлы сохранены:")
    for ds_id in results:
        print(f"  - cmapss_model_{ds_id}.h5")
        print(f"  - cmapss_metadata_{ds_id}.pkl")

#### 3.3. Выводы

1. __СЛОЖНОСТЬ ДАТАСЕТОВ__ (возрастает от FD001 к FD004):
   - FD001: Простейший (1 режим, 1 тип отказа)
   - FD002: Усложнен режимами работы
   - FD003: Усложнен множественными отказами
   - FD004: Наиболее сложный (множественные режимы + множественные отказы)
2. __ОЖИДАЕМОЕ КАЧЕСТВО ПРЕДСКАЗАНИЯ:__
   - Лучшие результаты на FD001 и FD003 (простые режимы)
   - Ухудшение на FD002 и FD004 из-за вариативности режимов работы
3. __ИНФОРМАТИВНЫЕ СЕНСОРЫ:__
   - Сенсоры с высокой корреляцией к деградации: 2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21
   - Некоторые сенсоры (1, 5, 6, 10, 16, 18, 19) малоинформативны
4. __РЕКОМЕНДАЦИИ:__
   - Использовать ансамбль моделей для улучшения результатов
   - Применять transfer learning от FD001 к более сложным датасетам
   - Добавить feature engineering (скорость изменения сенсоров)
   - Использовать сверточные слои для извлечения локальных паттернов